# HydrAI — PyTorch full axial PFR surrogate with Optuna

Purpose: train and evaluate a multi-output PyTorch MLP on every axial row of the processed dataset: inputs include run design columns plus `relative_position` (and `z_position_m` when present), matching the full-profile task in [Main_5_train_evaluate_tune_tree_model_evolution.ipynb](Main_5_train_evaluate_tune_tree_model_evolution.ipynb) §8. Train/test split is by simulation run (no leakage along a profile). Same physics targets as Main_5 full-profile.

Prerequisites: run Main_2 and Main_3 so `data/processed/features_targets_*.pkl` exists.

---

## Strategy

1. Features — `feature_cols` from Main_5 (includes `relative_position`; `run_cols` excludes axial columns for run IDs).
2. Scaling — `StandardScaler` on X and y (fit on training rows only) for comparable MSE across Pa vs mass fractions.
3. Optuna — optional TPE + median pruner (Section 6b); validation fold carved from **train** rows only (test **runs** held out in §4); objective = validation R² in physical units. Logs under `data/logs/`. Live plot: `python scripts/monitor/monitor_nn_training_progress.py` (set `LIVE=True` in that script to refresh).
4. Production training — `ReduceLROnPlateau` on periodic test R², early stopping, restore best test-R² weights (Section 8).
5. Memory — Section 2: `SUBSAMPLE_ROWS` / `SUBSAMPLE_MAX_ROWS` optionally cap train+test rows after the run split; `False` = full data.
6. Throughput — Section 2: `N_CPU_CORES` (cap PyTorch/BLAS threads), `OPTUNA_N_JOBS` (parallel §6b trials; keep `1` on one GPU), `USE_CUDA_AMP`, `USE_TORCH_COMPILE`. Section 1b enables TF32 + cuDNN benchmark when `device` is CUDA.

## Overfitting controls used here

- Dropout between hidden ReLUs (`nn.Dropout`, default `p=0.1` from config); active under `model.train()`, disabled under `model.eval()`. When `IF_HYPERPARAM_TUNING=True`, §6b may pick a different dropout within its search range.
- Held-out **test runs** — `train_test_split` on unique simulation **`run_id`** values (`test_size` in `configs/ml/main6_simplenn_config.json`); entire runs are held out so there is no train/test leakage along the same axial profile (§4).
- Scalers fit on training data only; the test split is only `transform`-ed (§5).
- Shuffled minibatches (`DataLoader(..., shuffle=True)`) for stochastic optimisation.
- Train vs test monitoring at periodic checkpoints (§8 / §8b) — a growing gap signals overfitting (first response: raise `dropout` or shrink `h1–h3` in config).

*Deliberately not enabled here:* Adam `weight_decay`, k-fold CV, data augmentation.

*Training (Section 8):* `ReduceLROnPlateau` on held-out **test** R² at checkpoints, early stopping when test R² stalls, then restore best test-R² weights before evaluation and export. Progress CSV under `data/logs/`; same monitor command as §6b (auto-switches to training curves). §8b still saves the final figure. See `docs/ML_CONFIG_GUIDE.md` for train / val / test splits.

---

Sections: 1 Setup · 2 Paths & flags · 3 Config & data · 4 Features & run-level split · 5 Scaling · 6 PyTorch MLP · 6b Optuna · 6b-ii Optuna plots · 6c Architecture · 7 Tensors · 8 Training · 8b Convergence · 9 Test metrics & axial overlays · 10–10c Parity / residuals (4-column grids, all targets) / R² bars · 11 Export


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ════════════════════════════════════════════════════════════════════════════
import os
import sys
import glob
import json
import re
import joblib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Project root
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle

setup_matplotlib()
start_run_log('Main_6_train_evaluate_SimpleNN_full_profile')
print("Libraries imported successfully.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1b. DEVICE (CPU / NVIDIA CUDA / Apple MPS)
# ════════════════════════════════════════════════════════════════════════════
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"PyTorch version: {torch.__version__}")
print(f"Using device:    {device}")
if device.type == "cuda":
    print(f"GPU name:        {torch.cuda.get_device_name(0)}")
    # Throughput: TF32 + cuDNN autotune (safe for fixed-ish MLP batch shapes).
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass
    print("CUDA: TF32 + cuDNN benchmark enabled for matmul throughput.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ════════════════════════════════════════════════════════════════════════════

# I/O flags
IF_PLOT_SHOWN          = True
IF_PLOT_EXPORT         = True
IF_MODEL_EXPORT        = True

# Training progress (§8): data/logs/ — same paths as scripts/monitor/monitor_nn_training_progress.py.
WRITE_TRAINING_PROGRESS_LOG = True
from src.utils.training_progress_log import (
    MAIN_6_STEM,
    optuna_snapshot_path,
    training_progress_log_path,
)

TRAINING_PROGRESS_LOG = training_progress_log_path(project_root, MAIN_6_STEM)

# Optional hyperparameter tuning (Section 6b). When True, runs Optuna TPE over
IF_HYPERPARAM_TUNING   = False

# --- Data size (row subsample after train/test split) ---
SUBSAMPLE_ROWS = True             # False = all profile rows

# Paths
CONFIG_PATH        = project_root / "configs" / "ml" / "main6_simplenn_config.json"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
EXPORT_DIR         = project_root / "models"
FIG_DIR            = project_root / "outputs" / "figures" / "Main_6_train_evaluate_SimpleNN_full_profile"

# Early partial config read for the "runtime" block only: CPU/subsample numbers
# are needed here to bootstrap thread pools before any data ops, ahead of
# Section 3's main config load (which reads neural_network.*/tuning.*).
_rt_cfg = {}
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as _f:
        _rt_cfg = json.load(_f).get("runtime", {})

SUBSAMPLE_MAX_ROWS = int(_rt_cfg.get("subsample_max_rows", 1_000))  # used only when SUBSAMPLE_ROWS is True
FULL_PROFILE_MAX_ROWS = int(SUBSAMPLE_MAX_ROWS) if SUBSAMPLE_ROWS else None
if SUBSAMPLE_ROWS:
    print(f"Data size: SUBSAMPLE_ROWS=True, max_rows={SUBSAMPLE_MAX_ROWS:,}")
else:
    print("Data size: SUBSAMPLE_ROWS=False (full production set)")

N_CPU_CORES = _rt_cfg.get("n_cpu_cores", 10)  # None = use all -//- Optuna
OPTUNA_N_JOBS = int(_rt_cfg.get("optuna_n_jobs", 10))

#___
from src.utils.cpu_threads import configure_cpu_threads, resolve_optuna_n_jobs

OPTUNA_N_JOBS = resolve_optuna_n_jobs(OPTUNA_N_JOBS, device.type, N_CPU_CORES)
# Bootstrap: full core budget for §3–§8 (§6b re-applies threads per trial; §8 cell also resets).
_CPU_THREAD_CFG = configure_cpu_threads(N_CPU_CORES, 1, device.type)
_OPTUNA_THREADS_PER_TRIAL = max(1, _CPU_THREAD_CFG["n_cores"] // max(1, int(OPTUNA_N_JOBS)))

# --- Training throughput (Section 8 + optional Section 6b) ---
# Automatic mixed precision on CUDA (Tensor Cores); no effect on CPU/MPS.
USE_CUDA_AMP = True
# torch.compile (PyTorch 2+): optional extra speed; first epoch pays compile cost.
USE_TORCH_COMPILE = False

# Plotting contract: convergence, parity, residuals, per-target R², and
# architecture figures must run correctly whether this flag is True or False.
# Optuna-specific PNGs live in Section 6b (run) and 6b-ii (plots from JSON only);
# with tuning off, 6b-ii prints [INFO] and skips — no impact on other plots.

# Architecture-visualisation flags (Section 6c). Both diagram styles can be
# toggled independently; the torchinfo summary requires `pip install torchinfo`
# and gracefully degrades to a plain `print(model)` if it is not installed.
IF_ARCH_DIAGRAM_MPL  = True   # matplotlib circles+lines diagram (themed by setup_matplotlib)
IF_ARCH_DIAGRAM_TIKZ = True   # write a standalone TikZ .tex file (compile externally with pdflatex)
IF_ARCH_SUMMARY      = True   # textual layer-by-layer summary via torchinfo

print(f"Plot: {IF_PLOT_SHOWN}  |  Export figs: {IF_PLOT_EXPORT}  |  Export model: {IF_MODEL_EXPORT}")
print(f"Optuna tuning (Section 6b): {'on' if IF_HYPERPARAM_TUNING else 'off'} - core plots work either way.")
print(
    f"CPU: N_CPU_CORES={N_CPU_CORES!r} -> {_CPU_THREAD_CFG['n_cores']} cores, "
    f"{_CPU_THREAD_CFG['torch_threads_per_job']} torch threads (§8 / prep)  |  "
    f"OPTUNA_N_JOBS={OPTUNA_N_JOBS} (~{_OPTUNA_THREADS_PER_TRIAL} threads/trial when §6b on)  |  "
    f"USE_CUDA_AMP={USE_CUDA_AMP}  |  USE_TORCH_COMPILE={USE_TORCH_COMPILE}"
)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD CONFIG & DATA
# ════════════════════════════════════════════════════════════════════════════

# ── When this notebook actually uses main6_simplenn_config.json ─────────────────
# This cell is the *only* place the JSON file is consulted; nothing else in
# Main_6 re-reads it. It runs on Kernel > Run All, or whenever you re-execute
# this single cell. Keys consumed by Main_6:
#   - top level:        test_size, random_state
#   - neural_network.*: epochs, batch_size, learning_rate, h1, h2, h3, dropout
# If the file is missing, or a key is omitted, the `config.get(..., default)`
# calls below fall back to the inline defaults shown as the second argument.
# Other keys in the JSON (random_forest, xgboost, ...) belong to Main_4/Main_5.
# and are ignored here.
# Edit main6_simplenn_config.json and re-run this cell to change behaviour;
# no kernel restart needed.

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
else:
    config = {}
    print(f"[WARN] Config not found: {CONFIG_PATH}. Using defaults.")

TEST_SIZE    = config.get("test_size", 0.2)
RANDOM_STATE = config.get("random_state", 42)

NN_CONFIG     = config.get("neural_network", {})
EPOCHS        = int(NN_CONFIG.get("epochs", 200))
BATCH_SIZE    = int(NN_CONFIG.get("batch_size", 256))
LEARNING_RATE = float(NN_CONFIG.get("learning_rate", 1e-3))
H1            = int(NN_CONFIG.get("h1", 128))
H2            = int(NN_CONFIG.get("h2", 64))
H3            = int(NN_CONFIG.get("h3", 32))
DROPOUT       = float(NN_CONFIG.get("dropout", 0.1))

#__ Optuna search controls (consumed in Section 6b only when IF_HYPERPARAM_TUNING=True).
TUNING_CONFIG    = NN_CONFIG.get("tuning", {})
N_TRIALS         = int(TUNING_CONFIG.get("n_trials", 30))
EPOCHS_PER_TRIAL = int(TUNING_CONFIG.get("epochs_per_trial", max(20, EPOCHS // 4)))
VAL_FRACTION     = float(TUNING_CONFIG.get("validation_fraction", 0.2))
TUNING_TIMEOUT_S = TUNING_CONFIG.get("timeout_seconds", None)

print(f"Split:    test_size={TEST_SIZE}, random_state={RANDOM_STATE}")
print(f"NN arch:  h1={H1}, h2={H2}, h3={H3}, dropout={DROPOUT}")
print(f"NN train: epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"Tuning:   enabled={IF_HYPERPARAM_TUNING}, n_trials={N_TRIALS}, "
      f"epochs/trial={EPOCHS_PER_TRIAL}, val_fraction={VAL_FRACTION}")

# Data (latest features_targets_*.pkl)
pkl_files = sorted(glob.glob(str(PROCESSED_DATA_DIR / "features_targets_*.pkl")), reverse=True)
if not pkl_files:
    raise FileNotFoundError(f"No features_targets_*.pkl in {PROCESSED_DATA_DIR}. Run Main_3 first.")

DATA_FILE = pkl_files[0]
loaded = load_portable_pickle(DATA_FILE)
df_features = loaded["df_features"]
df_target   = loaded["df_target"]

print(f"\nData: {Path(DATA_FILE).name}")
print(f"  Features: {df_features.shape[0]:,} rows × {df_features.shape[1]} cols")
print(f"  Targets:  {df_target.shape[0]:,} rows × {df_target.shape[1]} cols")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. FEATURES & TARGETS  (FULL AXIAL PROFILE; includes relative_position)
# ════════════════════════════════════════════════════════════════════════════
# One row per axial location. Inputs = run design + spatial coordinates so the MLP
# learns PFR evolution. Train/test split is by simulation run (below) so profiles
# are not split across train and test.

feature_cols = [
    "initial_temperature_K", "initial_pressure_Pa",
    "reactor_length_m", "reactor_diameter_m",
    "mass_flow_rate_kgps", "heat_flux_Wm2",
    "z_position_m", "relative_position",
]
if "reactant_type" in df_features.columns:
    feature_cols.append("reactant_type")
feature_cols = [c for c in feature_cols if c in df_features.columns]

if "relative_position" not in feature_cols:
    raise KeyError(
        "Full-profile NN requires column 'relative_position' in df_features. "
        "Re-run upstream preprocessing if missing."
    )

run_cols = [c for c in feature_cols if c not in ("z_position_m", "relative_position")]

primary_targets = [
    "temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3",
    "mean_molecular_weight_kgkmol",
    "heat_capacity_cp_JkgK", "heat_capacity_cv_JkgK",
    "enthalpy_Jkg", "thermal_conductivity_WmK",
]
state_target_cols = [c for c in primary_targets if c in df_target.columns]

_lump_cols = [c for c in df_target.columns if c.startswith("Y_lump_")]
species_cols = sorted(_lump_cols) if _lump_cols else [c for c in df_target.columns if c.startswith("Y_")]

target_cols = state_target_cols + species_cols

# ── Chemistry grouping (for optional species charts / manifest metadata) ─────
def classify_species_chemistry(species_name):
    name = species_name[2:] if species_name.startswith("Y_") else species_name
    name_base = name.split("(")[0]

    if name_base == "H2":
        return "hydrogen"
    if name_base in {"Water", "Ar", "He", "Ne", "N2", "H"}:
        return "diluent"
    if name_base == "C6H14":
        return "feedstock"
    if name_base in {"C2H4", "C3H6", "C4H6", "C4H8"}:
        return "olefins"
    if name_base in {"Benzene", "Toluene", "Styrene", "C8H10"}:
        return "aromatics"
    if name_base in {"CH4", "CC", "CCC"}:
        return "paraffins"
    if name_base.startswith("C#C") or name_base == "C2H2":
        return "coke_precursors"
    if name_base in {"C5H6", "C5H5", "C3H4", "C4H4", "C4H5"}:
        return "coke_precursors"
    match = re.match(r"^C(\d+)H(\d+)", name_base)
    if match:
        c, h = int(match.group(1)), int(match.group(2))
        if c >= 6 and h / c < 1.5:
            return "coke_precursors"
    if name_base in {
        "CH3", "C2H3", "C2H5", "C3H5", "C3H7", "C4H7", "C4H9",
        "C5H7", "C5H9", "C5H11", "C6H7", "C6H9", "C6H11", "C6H13",
        "C7H9", "C7H11", "C3H3",
    }:
        return "radicals"
    return "other"


chemistry_groups = {}
if species_cols and all(c.startswith("Y_lump_chem_") for c in species_cols):
    for c in species_cols:
        role = c[len("Y_lump_chem_") :]
        chemistry_groups[role] = [c]
elif species_cols and all(c.startswith("Y_lump_carbon_") for c in species_cols):
    for c in species_cols:
        suf = c[len("Y_lump_carbon_") :]
        role = "inert" if str(suf).lower() == "inert" else f"carbon_{suf}"
        chemistry_groups[role] = [c]
else:
    for col in species_cols:
        role = classify_species_chemistry(col)
        chemistry_groups.setdefault(role, []).append(col)

print("Species / lump columns by chemistry role:")
for role, cols in sorted(chemistry_groups.items()):
    print(f"  {role}: {len(cols)} column(s)")

X_full = df_features.loc[:, feature_cols].copy()
y_full = df_target.loc[:, target_cols].copy()

label_encoder = None
if "reactant_type" in X_full.columns:
    label_encoder = LabelEncoder()
    X_full["reactant_type"] = label_encoder.fit_transform(X_full["reactant_type"].astype(str))

full_run_id = df_features.groupby(run_cols, dropna=False).ngroup()
unique_runs = np.array(sorted(pd.unique(full_run_id)))
train_runs, test_runs = train_test_split(
    unique_runs,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
train_mask = full_run_id.isin(train_runs)
test_mask = full_run_id.isin(test_runs)

X_train_df = X_full.loc[train_mask].copy()
X_test_df = X_full.loc[test_mask].copy()
y_train_df = y_full.loc[train_mask].copy()
y_test_df = y_full.loc[test_mask].copy()

if FULL_PROFILE_MAX_ROWS is not None and len(X_train_df) + len(X_test_df) > FULL_PROFILE_MAX_ROWS:
    n_train_keep = int(FULL_PROFILE_MAX_ROWS * (1 - TEST_SIZE))
    n_test_keep = FULL_PROFILE_MAX_ROWS - n_train_keep
    X_train_df = X_train_df.sample(n=min(n_train_keep, len(X_train_df)), random_state=RANDOM_STATE)
    y_train_df = y_train_df.loc[X_train_df.index]
    X_test_df = X_test_df.sample(n=min(n_test_keep, len(X_test_df)), random_state=RANDOM_STATE)
    y_test_df = y_test_df.loc[X_test_df.index]
    print(
        f"[INFO] Subsampled full-profile rows to {len(X_train_df) + len(X_test_df):,} "
        f"(FULL_PROFILE_MAX_ROWS={FULL_PROFILE_MAX_ROWS})"
    )

print(f"\nFull-profile dataset: {len(X_full):,} total rows")
print(f"  Train rows / runs: {len(X_train_df):,} / {len(train_runs):,}")
print(f"  Test  rows / runs: {len(X_test_df):,} / {len(test_runs):,}")
print(f"  Features ({len(feature_cols)}): {feature_cols}")
print(f"  State targets ({len(state_target_cols)}): {state_target_cols}")
print(f"  Species / lump targets: {len(species_cols)} columns")
print(f"  Total targets: {len(target_cols)}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. SCALING  (train rows only; no test leakage)
# ════════════════════════════════════════════════════════════════════════════
# StandardScaler on X and y. Predictions are inverse-
# transformed to physical units for metrics, plots, and Optuna objective.

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train_df)
X_test_s = scaler_X.transform(X_test_df)

SCALE_TARGETS = True
if SCALE_TARGETS:
    scaler_y = StandardScaler()
    y_train_s = scaler_y.fit_transform(y_train_df)
    y_test_s = scaler_y.transform(y_test_df)
else:
    scaler_y = None
    y_train_s = y_train_df.values
    y_test_s = y_test_df.values

print(
    f"Train: {len(X_train_df):,} rows ({100 * (1 - TEST_SIZE):.0f}% of runs)  |  "
    f"Test: {len(X_test_df):,} rows ({100 * TEST_SIZE:.0f}% of runs)"
)
print(f"X shape: {X_train_s.shape}  |  y shape: {y_train_s.shape}  |  scale_y={SCALE_TARGETS}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. PYTORCH MULTI-OUTPUT REGRESSION MLP
# ════════════════════════════════════════════════════════════════════════════
# Architecture defined in src/models/simple_nn.py (shared with Main_7).
# Dropout is active under model.train() and disabled by model.eval(), so
# test-time predictions are deterministic.

from src.models import SimpleNN

torch.manual_seed(RANDOM_STATE)

n_inputs  = X_train_s.shape[1]
n_outputs = y_train_s.shape[1]
model = SimpleNN(n_inputs, H1, H2, H3, n_outputs, dropout=DROPOUT).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")
print(f"Inputs: {n_inputs}  (includes z_position_m and relative_position)")
print(f"Outputs: {n_outputs}  ({len(state_target_cols)} state + {len(species_cols)} species)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6b. OPTIONAL OPTUNA HYPERPARAMETER TUNING
# ════════════════════════════════════════════════════════════════════════════
# Gated entirely by the IF_HYPERPARAM_TUNING flag (Section 2). When True:
#   1. Carve a validation fold out of the *training* split (test set is left
#      completely untouched).
#   2. Run an Optuna TPE study over h1, h2, h3, dropout, learning_rate,
#      batch_size; the objective is validation R² in physical units.
#   3. Median pruner stops bad trials early.
#   4. Pick the best trial, refit `model` with those hyperparameters and
#      overwrite the notebook-level H1/H2/H3/DROPOUT/LEARNING_RATE/BATCH_SIZE
#      so the rest of the notebook (tensors, training loop, evaluation,
#      export) runs unchanged on the tuned net.
#   5. Writes optuna JSON under data/logs/; Section 6b-ii (next cell)
#      reads that file so you can refresh figures without re-running the study.
# When False: the cell prints one line and skips. No new dependency is loaded.
#
# Install once:  pip install optuna

best_params       = None
best_val_r2       = None
tuning_study      = None
n_trials_complete = 0

if IF_HYPERPARAM_TUNING:
    try:
        import optuna
        from optuna.exceptions import TrialPruned
    except ImportError as exc:
        raise ImportError(
            "IF_HYPERPARAM_TUNING=True requires Optuna. Install with `pip install optuna`."
        ) from exc

    from torch import amp as _torch_amp

    _use_amp_tune = bool(USE_CUDA_AMP) and device.type == "cuda"

    # ── Validation fold (train -> train' / val); test stays held out ──────────
    rng_idx = np.random.default_rng(RANDOM_STATE)
    n_train = X_train_s.shape[0]
    perm    = rng_idx.permutation(n_train)
    n_val   = int(round(VAL_FRACTION * n_train))
    val_idx = perm[:n_val]
    tr_idx  = perm[n_val:]

    Xtr_t = torch.tensor(X_train_s[tr_idx],  dtype=torch.float32, device=device)
    ytr_t = torch.tensor(y_train_s[tr_idx],  dtype=torch.float32, device=device)
    Xva_t = torch.tensor(X_train_s[val_idx], dtype=torch.float32, device=device)
    yva_physical = y_train_df.values[val_idx]

    # ── Objective: validation R² (physical units, uniform avg across targets) ──
    def _tune_objective(trial):
        configure_cpu_threads(N_CPU_CORES, OPTUNA_N_JOBS, device.type)
        h1_t  = trial.suggest_int("h1", 32, 256, step=32)
        h2_t  = trial.suggest_int("h2", 16, 128, step=16)
        h3_t  = trial.suggest_int("h3", 8,  64,  step=8)
        dp_t  = trial.suggest_float("dropout", 0.0, 0.3)
        lr_t  = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        bs_t  = trial.suggest_categorical("batch_size", [64, 128, 256, 512])

        torch.manual_seed(RANDOM_STATE)
        net = SimpleNN(n_inputs, h1_t, h2_t, h3_t, n_outputs, dropout=dp_t).to(device)
        opt = optim.Adam(net.parameters(), lr=lr_t)
        ds  = TensorDataset(Xtr_t, ytr_t)
        loader = DataLoader(ds, batch_size=min(bs_t, len(ds)), shuffle=True)

        report_every = max(1, EPOCHS_PER_TRIAL // 5)
        scaler_t = _torch_amp.GradScaler("cuda", enabled=_use_amp_tune)
        for ep in range(EPOCHS_PER_TRIAL):
            net.train()
            for xb, yb in loader:
                opt.zero_grad(set_to_none=True)
                if _use_amp_tune:
                    with _torch_amp.autocast("cuda"):
                        loss = F.mse_loss(net(xb), yb)
                    scaler_t.scale(loss).backward()
                    scaler_t.step(opt)
                    scaler_t.update()
                else:
                    loss = F.mse_loss(net(xb), yb)
                    loss.backward()
                    opt.step()

            if ep % report_every == 0 or ep == EPOCHS_PER_TRIAL - 1:
                net.eval()
                with torch.no_grad():
                    if _use_amp_tune:
                        with _torch_amp.autocast("cuda"):
                            yp_s = net(Xva_t).cpu().numpy()
                    else:
                        yp_s = net(Xva_t).cpu().numpy()
                yp = scaler_y.inverse_transform(yp_s) if scaler_y is not None else yp_s
                val_r2 = r2_score(yva_physical, yp, multioutput="uniform_average")
                trial.report(val_r2, ep)
                if trial.should_prune():
                    raise TrialPruned()

        return val_r2

    # ── Incremental JSON for external Optuna viewer (Section 6b-ii reads final file)
    from src.utils.training_progress_log import init_optuna_snapshot, write_optuna_snapshot

    _optuna_snap_path = optuna_snapshot_path(project_root, MAIN_6_STEM)
    _optuna_snap_path.parent.mkdir(parents=True, exist_ok=True)
    init_optuna_snapshot(
        _optuna_snap_path,
        study_name="Main_6_SimpleNN_full_profile_tuning",
    )
    print(f"Cleared Optuna monitor log for new study: {_optuna_snap_path}")

    def _optuna_progress_callback(study, trial):
        if trial.value is None:
            return
        write_optuna_snapshot(_optuna_snap_path, study)

    # ── Run study ─────────────────────────────────────────────────────────────
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    pruner  = optuna.pruners.MedianPruner(n_warmup_steps=2)
    tuning_study = optuna.create_study(
        direction="maximize", sampler=sampler, pruner=pruner,
        study_name="Main_6_SimpleNN_full_profile_tuning",
    )
    tuning_study.optimize(
        _tune_objective, n_trials=N_TRIALS, timeout=TUNING_TIMEOUT_S,
        show_progress_bar=False,
        n_jobs=int(OPTUNA_N_JOBS),
        callbacks=[_optuna_progress_callback],
    )

    completed = [t for t in tuning_study.trials if t.value is not None]
    n_trials_complete = len(completed)
    best_params = tuning_study.best_params
    best_val_r2 = float(tuning_study.best_value)

    print("=" * 70)
    print(f"Optuna tuning finished  ·  {n_trials_complete} completed trials")
    print(f"  Best validation R² (uniform avg, physical units): {best_val_r2:.4f}")
    print(f"  Best params: {best_params}")
    print("=" * 70)

    # ── Adopt the best hyperparameters and rebuild the production model ───────
    H1            = int(best_params.get("h1", H1))
    H2            = int(best_params.get("h2", H2))
    H3            = int(best_params.get("h3", H3))
    DROPOUT       = float(best_params.get("dropout", DROPOUT))
    LEARNING_RATE = float(best_params.get("learning_rate", LEARNING_RATE))
    BATCH_SIZE    = int(best_params.get("batch_size", BATCH_SIZE))

    torch.manual_seed(RANDOM_STATE)
    model = SimpleNN(n_inputs, H1, H2, H3, n_outputs, dropout=DROPOUT).to(device)
    print(f"\nRebuilt model with tuned params:")
    print(f"  arch:  h1={H1}, h2={H2}, h3={H3}, dropout={DROPOUT}")
    print(f"  train: lr={LEARNING_RATE}, batch_size={BATCH_SIZE}")

    # ── Final snapshot for Section 6b-ii (importances when available) ─────────
    _imp = None
    try:
        _imp = dict(optuna.importance.get_param_importances(tuning_study))
    except Exception as _e_imp:
        print(f"[INFO] Param importances not saved for plot cell: {_e_imp}")
    write_optuna_snapshot(_optuna_snap_path, tuning_study, importances=_imp)
    print(f"  Optuna plot snapshot: {_optuna_snap_path}")

else:
    print("Optuna hyperparameter tuning disabled (IF_HYPERPARAM_TUNING=False).")
    print("Set IF_HYPERPARAM_TUNING=True in Section 2 to enable a TPE search.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6b-ii. OPTUNA DIAGNOSTIC PLOTS (run separately from Section 6b)
# ════════════════════════════════════════════════════════════════════════════
# Reads optuna_tuning_plot_data.json written when Section 6b completes with
# IF_HYPERPARAM_TUNING=True. Re-run this cell to refresh figures without
# re-running the study. Writes optuna_optimization_history.png,
# optuna_parallel_coordinate.png, and optuna_param_importance.png (when
# importances are present). Requires FIG_DIR from Section 2 and prior imports
# (numpy, matplotlib, json).

_snap_path = optuna_snapshot_path(project_root, MAIN_6_STEM)
if not _snap_path.exists():
    print(
        "[INFO] No Optuna plot snapshot. Run Section 6b with IF_HYPERPARAM_TUNING=True first "
        f"(writes {_snap_path.name})."
    )
else:
    with open(_snap_path, encoding="utf-8") as _f:
        _optuna_plot = json.load(_f)

    _trials = _optuna_plot.get("trials", [])
    best_val_r2 = float(_optuna_plot.get("best_val_r2", float("nan")))
    n_trials_complete = int(_optuna_plot.get("n_trials_complete", len(_trials)))

    trial_nums = [t["number"] for t in _trials]
    trial_vals = [t["value"] for t in _trials]
    best_so_far = np.maximum.accumulate(trial_vals) if trial_vals else []

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(trial_nums, trial_vals, "o", alpha=0.55, color="b", label="Trial val R²")
    if len(best_so_far) > 0:
        ax.plot(trial_nums, best_so_far, "-", lw=1.6, color="r", label="Best so far")
    ax.set_xlabel("Trial")
    ax.set_ylabel("Validation R² (uniform avg)")
    ax.set_title(
        f"Optuna optimisation history  ·  {n_trials_complete} trials"
        f"  ·  best R²={best_val_r2:.4f}"
    )
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / "optuna_optimization_history.png", dpi=150, bbox_inches="tight")
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)

    # ── Parallel coordinates: tested hyperparameter tuples vs validation R² ───
    _rows_pc = []
    for t in _trials:
        p = t.get("params")
        if not isinstance(p, dict) or not p:
            continue
        _need = ("h1", "h2", "h3", "dropout", "learning_rate", "batch_size")
        if not all(k in p for k in _need):
            continue
        _rows_pc.append([float(p[k]) for k in _need] + [float(t["value"])])

    if len(_rows_pc) >= 1:
        Xp = np.asarray(_rows_pc, dtype=float)
        _col_labels = ["h1", "h2", "h3", "dropout", "learning_rate", "batch_size", "val R²"]
        n_dim = Xp.shape[1]
        Xn = np.zeros_like(Xp)
        _lr_i = 4
        for j in range(n_dim):
            col = Xp[:, j].copy()
            if j == _lr_i:
                col = np.log10(np.maximum(col, 1e-12))
            lo, hi = float(np.nanmin(col)), float(np.nanmax(col))
            if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
                Xn[:, j] = 0.5
            else:
                Xn[:, j] = (col - lo) / (hi - lo)
        ix = np.arange(n_dim)
        obj = Xp[:, -1]
        _vmin, _vmax = float(np.nanmin(obj)), float(np.nanmax(obj))
        _norm_pc = plt.Normalize(vmin=_vmin, vmax=_vmax)
        _cmap_pc = plt.cm.turbo
        if not np.isfinite(_vmin) or not np.isfinite(_vmax) or _vmax <= _vmin:
            _norm_pc = plt.Normalize(0.0, 1.0)
        fig_pc, ax_pc = plt.subplots(figsize=(11, 5.0))
        ibest = int(np.nanargmax(obj))
        for i in range(len(Xn)):
            if i == ibest:
                continue
            ax_pc.plot(
                ix,
                Xn[i],
                color=_cmap_pc(_norm_pc(obj[i])),
                alpha=0.38,
                lw=1.1,
                zorder=1,
            )
        ax_pc.plot(ix, Xn[ibest], color="0.15", lw=1.5, zorder=3, label="Best trial")
        ax_pc.set_xticks(ix)
        ax_pc.set_xticklabels(_col_labels, rotation=90, ha="right")
        ax_pc.set_ylabel("Normalized (min–max per axis; log for learning rate)")
        ax_pc.set_ylim(-0.06, 1.06)
        ax_pc.set_title("Parallel coordinates — tested hyperparameters vs validation R²")
        ax_pc.grid(True, axis="y", alpha=0.25)
        ax_pc.legend(loc="lower right", fontsize=8)
        sm_pc = plt.cm.ScalarMappable(norm=_norm_pc, cmap=_cmap_pc)
        sm_pc.set_array([])
        fig_pc.colorbar(
            sm_pc,
            ax=ax_pc,
            fraction=0.046,
            pad=0.02,
            label="Validation R² (uniform avg)",
        )
        plt.tight_layout()
        if IF_PLOT_EXPORT:
            fig_pc.savefig(
                FIG_DIR / "optuna_parallel_coordinate.png",
                dpi=150,
                bbox_inches="tight",
            )
        if IF_PLOT_SHOWN:
            plt.show()
        plt.close(fig_pc)
    else:
        print(
            "[INFO] No per-trial params in snapshot; parallel coordinates skipped. "
            "Re-run Section 6b once to refresh optuna_tuning_plot_data.json."
        )

    _imp = _optuna_plot.get("importances")
    if _imp:
        fig2, ax2 = plt.subplots(figsize=(11, 3.6))
        names = list(_imp.keys())
        vals = list(_imp.values())
        ax2.barh(
            names[::-1],
            vals[::-1],
            facecolor="white",
            edgecolor="blue",
            linewidth=0.65,
            hatch="///",
        )
        ax2.set_xlabel("Importance (fANOVA)")
        ax2.set_title("Optuna parameter importance (validation R²)")
        ax2.grid(True, axis="x", alpha=0.3)
        plt.tight_layout()
        if IF_PLOT_EXPORT:
            fig2.savefig(FIG_DIR / "optuna_param_importance.png", dpi=150, bbox_inches="tight")
        if IF_PLOT_SHOWN:
            plt.show()
        plt.close(fig2)
    else:
        print("[INFO] No importances in snapshot; skipping parameter-importance figure.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6c. ARCHITECTURE VISUALISATION
# ════════════════════════════════════════════════════════════════════════════
# Three independent toggles (see Section 2 paths/flags cell):
#   IF_ARCH_SUMMARY      → torchinfo textual summary (layer-by-layer shapes/params)
#   IF_ARCH_DIAGRAM_MPL  → matplotlib schematic  (src.utils.draw_mlp_architecture)
#   IF_ARCH_DIAGRAM_TIKZ → standalone TikZ .tex  (src.utils.write_tikz_mlp)
# The diagram renderers live in src/utils/plot_nn_architecture.py so this cell
# only has to describe the network and call them.
from src.utils import draw_mlp_architecture, write_tikz_mlp

layer_sizes = [n_inputs, H1, H2, H3, n_outputs]
layer_names = ['Input', 'Hidden 1', 'Hidden 2', 'Hidden 3', 'Output']
layer_ops   = ['Linear + ReLU + Dropout',
               'Linear + ReLU + Dropout',
               'Linear + ReLU + Dropout',
               'Linear']

# ── (a) torchinfo textual summary ─────────────────────────────────────────────
if IF_ARCH_SUMMARY:
    try:
        from torchinfo import summary as _torchinfo_summary
        print(_torchinfo_summary(
            model, input_size=(1, n_inputs), device=device,
            col_names=("input_size", "output_size", "num_params", "trainable"),
            verbose=0,
        ))
    except ImportError:
        print("[INFO] torchinfo not installed. `pip install torchinfo` for a layer table.")
        print(model)

# ── (b) Matplotlib architecture diagram ──────────────────────────────────────
if IF_ARCH_DIAGRAM_MPL:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    save_path = FIG_DIR / 'architecture_diagram.png' if IF_PLOT_EXPORT else None
    fig_arch = draw_mlp_architecture(
        layer_sizes, layer_names, layer_ops, DROPOUT,
        save_path=save_path,
    )
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig_arch)

# ── (c) Publication-grade TikZ (.tex) ────────────────────────────────────────
if IF_ARCH_DIAGRAM_TIKZ:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    tex_path = write_tikz_mlp(
        layer_sizes, layer_names, layer_ops, DROPOUT,
        FIG_DIR / 'architecture_diagram.tex',
    )
    print(f"\nTikZ source written: {tex_path}")
    print("  Compile with:  pdflatex architecture_diagram.tex")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. TENSORS & DATA LOADERS
# ════════════════════════════════════════════════════════════════════════════
X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=device)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test_s,  dtype=torch.float32, device=device)

train_ds     = TensorDataset(X_train_t, y_train_t)
batch_size   = min(BATCH_SIZE, len(train_ds))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

print(f"Tensors on device: {device}")
print(f"X_train_t: {tuple(X_train_t.shape)}  |  y_train_t: {tuple(y_train_t.shape)}")
print(f"Batches/epoch: {len(train_loader)}  |  batch_size: {batch_size}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. TRAINING LOOP
# ════════════════════════════════════════════════════════════════════════════
_CPU_THREAD_CFG = configure_cpu_threads(N_CPU_CORES, 1, device.type)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4, threshold=1e-4, min_lr=1e-6
)

from torch import amp

_use_amp = bool(USE_CUDA_AMP) and device.type == "cuda"
_scaler = amp.GradScaler("cuda", enabled=_use_amp)

if USE_TORCH_COMPILE and hasattr(torch, "compile"):
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("[INFO] torch.compile(model) enabled (PyTorch 2+).")
    except Exception as _e_tc:
        print(f"[INFO] torch.compile skipped: {_e_tc}")

# Per-epoch training MSE (standardised target space) and periodic test
# checkpoints with MSE + uniform-average R² in physical units. The R²
# checkpoints are cheap (single forward pass on the full test tensor) and
# give the convergence plot real information instead of just "loss went down".
# This is also the main *overfitting diagnostic* in this notebook: compare
# train vs test curves in §8b (see overview markdown "Overfitting controls used here").
#
# Also at each checkpoint: ReduceLROnPlateau steps on test R², we track the best
# test R² snapshot and optionally early-stop if test R² stalls; then we restore
# the best weights before Section 9 evaluation / export.
MIN_DELTA_R2 = 1e-4
EARLY_STOP_PATIENCE_EVALS = 12  # consecutive checkpoints without test R² gain

train_loss_log = []
epoch_log      = []
test_loss_log  = []
train_r2_log   = []
test_r2_log    = []

best_state_cpu = None
best_te_r2     = float("-inf")
best_epoch     = 0
stall_evals    = 0
EARLY_STOPPED  = False

from src.utils.training_progress_log import (
    append_training_progress,
    init_training_progress_log,
)

if WRITE_TRAINING_PROGRESS_LOG:
    init_training_progress_log(TRAINING_PROGRESS_LOG)
    print(f"Training progress log: {TRAINING_PROGRESS_LOG}")

eval_every = max(1, EPOCHS // 40)
log_every  = max(1, EPOCHS // 20)

y_train_true = y_train_df.values
y_test_true  = y_test_df.values

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        optimizer.zero_grad(set_to_none=True)
        if _use_amp:
            with amp.autocast("cuda"):
                y_pred = model(xb)
                loss = criterion(y_pred, yb)
            _scaler.scale(loss).backward()
            _scaler.step(optimizer)
            _scaler.update()
        else:
            y_pred = model(xb)
            loss = criterion(y_pred, yb)
            loss.backward()
            optimizer.step()
        batch_losses.append(loss.item())

    mean_train_loss = float(np.mean(batch_losses))
    train_loss_log.append(mean_train_loss)

    if epoch % eval_every == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            if _use_amp:
                with amp.autocast("cuda"):
                    y_pred_train_t = model(X_train_t)
                    y_pred_test_t = model(X_test_t)
            else:
                y_pred_train_t = model(X_train_t)
                y_pred_test_t = model(X_test_t)
            test_mse = criterion(y_pred_test_t, y_test_t).item()

            ypt = y_pred_train_t.cpu().numpy()
            yps = y_pred_test_t.cpu().numpy()
            if scaler_y is not None:
                ypt = scaler_y.inverse_transform(ypt)
                yps = scaler_y.inverse_transform(yps)

            tr_r2 = r2_score(y_train_true, ypt, multioutput='uniform_average')
            te_r2 = r2_score(y_test_true,  yps, multioutput='uniform_average')

        scheduler.step(te_r2)

        epoch_log.append(epoch)
        test_loss_log.append(test_mse)
        train_r2_log.append(tr_r2)
        test_r2_log.append(te_r2)

        if WRITE_TRAINING_PROGRESS_LOG:
            lr_cur = optimizer.param_groups[0]["lr"]
            append_training_progress(
                TRAINING_PROGRESS_LOG,
                epoch,
                mean_train_loss,
                test_mse=test_mse,
                train_r2=tr_r2,
                test_r2=te_r2,
                lr=lr_cur,
                is_checkpoint=True,
            )

        if te_r2 > best_te_r2 + MIN_DELTA_R2:
            best_te_r2 = float(te_r2)
            best_epoch = int(epoch)
            best_state_cpu = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stall_evals = 0
        else:
            stall_evals += 1

        if stall_evals >= EARLY_STOP_PATIENCE_EVALS:
            EARLY_STOPPED = True
            print(
                f"\nEarly stopping: no test R² improvement (>{MIN_DELTA_R2}) for "
                f"{EARLY_STOP_PATIENCE_EVALS} consecutive checkpoints "
                f"(last eval epoch {epoch})."
            )
            break

    elif WRITE_TRAINING_PROGRESS_LOG:
        append_training_progress(
            TRAINING_PROGRESS_LOG,
            epoch,
            mean_train_loss,
            is_checkpoint=False,
        )

    if epoch % log_every == 0 or epoch == EPOCHS - 1:
        msg = f"Epoch {epoch:4d}/{EPOCHS} | train MSE: {mean_train_loss:.6f}"
        if test_loss_log:
            lr_cur = optimizer.param_groups[0]["lr"]
            msg += (
                f" | test MSE: {test_loss_log[-1]:.6f} | test R²: {test_r2_log[-1]:.4f}"
                f" | lr: {lr_cur:.2e}"
            )
        print(msg)


if best_state_cpu is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state_cpu.items()})
    print(
        f"\nRestored best weights from epoch {best_epoch} "
        f"(best checkpoint test R²={best_te_r2:.4f})."
    )
else:
    best_te_r2 = float(test_r2_log[-1]) if test_r2_log else float("nan")
    best_epoch = int(epoch_log[-1]) if epoch_log else -1

BEST_TEST_R2_CKPT = float(best_te_r2)
BEST_EPOCH_CKPT = int(best_epoch)

print("\nTraining complete.")
print(
    f"  early_stopped={EARLY_STOPPED}  |  best checkpoint test R²={BEST_TEST_R2_CKPT:.4f} "
    f"@ epoch {BEST_EPOCH_CKPT}"
)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8b. CONVERGENCE — TRAIN/TEST MSE AND R² VS EPOCH
# ════════════════════════════════════════════════════════════════════════════
fig, (ax_loss, ax_r2) = plt.subplots(1, 2, figsize=(13, 4.5))

# ── Panel A: MSE loss vs epoch ────────────────────────────────────────────────
ax_loss.plot(range(len(train_loss_log)), train_loss_log,
             color='b', lw=1.6, label='Train (per epoch)')
ax_loss.plot(epoch_log, test_loss_log,
             color='r', lw=1.6, ls='-', ms=4, label='Test (checkpoints)')
loss_label = 'MSE (standardised targets)' if SCALE_TARGETS else 'MSE'
ax_loss.set_xlabel('Epoch')
ax_loss.set_ylabel(loss_label)
ax_loss.set_yscale('log')
ax_loss.set_title('Convergence — MSE loss')
ax_loss.grid(True, which='both', alpha=0.35)
ax_loss.legend(loc='upper right', frameon=True, fontsize=9)

# ── Panel B: R² vs epoch (physical units, uniform avg) ────────────────────────
ax_r2.plot(epoch_log, train_r2_log,
           color='b', lw=1.6, label='Train')
ax_r2.plot(epoch_log, test_r2_log,
           color='r', lw=1.6, ls='-', label='Test')
ax_r2.axhline(1.0, color='k', lw=0.8, alpha=0.4)
ax_r2.axhline(0.0, color='k', lw=0.8, alpha=0.2)
ax_r2.set_xlabel('Epoch')
ax_r2.set_ylabel('R² (uniform average over targets)')
ax_r2.set_title('Convergence — R² (physical units)')
r2_min = min(min(train_r2_log), min(test_r2_log))
ax_r2.set_ylim(bottom=min(-0.05, r2_min - 0.05), top=1.02)
ax_r2.grid(True, alpha=0.35)
ax_r2.legend(loc='lower right', frameon=True, fontsize=9)

# Final overfit summary as a figure annotation (curves use last checkpoint values;
# best checkpoint test R² is from the weight snapshot restored at end of §8).
final_gap = train_r2_log[-1] - test_r2_log[-1]
fig.suptitle(
    f"Training convergence  |  final-epoch  train R²={train_r2_log[-1]:.4f},  "
    f"test R²={test_r2_log[-1]:.4f},  gap={final_gap:+.4f}  |  "
    f"best test R²={BEST_TEST_R2_CKPT:.4f} @ ep {BEST_EPOCH_CKPT}",
    y=1.02, fontsize=11,
)
plt.tight_layout()

if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'training_convergence.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9. TEST-SET EVALUATION
# ════════════════════════════════════════════════════════════════════════════
# Predictions: tensors → inverse transform to physical units when SCALE_TARGETS.
# When IF_MODEL_EXPORT: writes per-target + 4-field group metric CSVs under EXPORT_DIR
# (same filenames as §11). Parity/residual figure grids use 4 columns in §10 / §10b.

from torch import amp as _amp_eval

_eval_amp = bool(USE_CUDA_AMP) and device.type == "cuda"
model.eval()
with torch.no_grad():
    if _eval_amp:
        with _amp_eval.autocast("cuda"):
            y_pred_train_s = model(X_train_t).cpu().numpy()
            y_pred_test_s = model(X_test_t).cpu().numpy()
    else:
        y_pred_train_s = model(X_train_t).cpu().numpy()
        y_pred_test_s = model(X_test_t).cpu().numpy()

if scaler_y is not None:
    y_pred_train = scaler_y.inverse_transform(y_pred_train_s)
    y_pred_test  = scaler_y.inverse_transform(y_pred_test_s)
else:
    y_pred_train = y_pred_train_s
    y_pred_test  = y_pred_test_s

from src.utils.profile_predictions import anchor_inlet_profile_predictions

y_pred_test, _n_inlet_anchored = anchor_inlet_profile_predictions(
    y_pred_test,
    y_test_df.values,
    full_run_id.loc[X_test_df.index].to_numpy(),
    X_test_df["relative_position"].to_numpy(),
)
print(
    f"Inlet BC anchoring (test): {_n_inlet_anchored} row(s) at min x/L per run "
    f"— axial overlays use the same Cantera inlet state as ML inputs."
)

y_train_np = y_train_df.values
y_test_np  = y_test_df.values

train_r2 = r2_score(y_train_np, y_pred_train, multioutput='uniform_average')
test_r2  = r2_score(y_test_np,  y_pred_test,  multioutput='uniform_average')
test_mae  = mean_absolute_error(y_test_np, y_pred_test, multioutput='uniform_average')
test_rmse = np.sqrt(mean_squared_error(y_test_np, y_pred_test, multioutput='uniform_average'))

# Uniform-average R² within families (same column grouping as Main_4: state/thermo vs species lumps)
_col_index = {name: j for j, name in enumerate(target_cols)}
_j_state = [_col_index[c] for c in state_target_cols]
_j_species = [_col_index[c] for c in species_cols]
train_r2_state = r2_score(y_train_np[:, _j_state], y_pred_train[:, _j_state], multioutput="uniform_average")
test_r2_state = r2_score(y_test_np[:, _j_state], y_pred_test[:, _j_state], multioutput="uniform_average")
train_r2_species = r2_score(y_train_np[:, _j_species], y_pred_train[:, _j_species], multioutput="uniform_average")
test_r2_species = r2_score(y_test_np[:, _j_species], y_pred_test[:, _j_species], multioutput="uniform_average")

print("=" * 70)
print("PyTorch MLP — Test set summary")
print("=" * 70)
print(f"  Train R² (uniform avg): {train_r2:.4f}")
print(f"  Test  R² (uniform avg): {test_r2:.4f}")
print(f"  Train R² state/thermo ({len(state_target_cols)} cols, uniform avg): {train_r2_state:.4f}")
print(f"  Test  R² state/thermo ({len(state_target_cols)} cols, uniform avg): {test_r2_state:.4f}")
print(f"  Train R² species lumps ({len(species_cols)} cols, uniform avg): {train_r2_species:.4f}")
print(f"  Test  R² species lumps ({len(species_cols)} cols, uniform avg): {test_r2_species:.4f}")
print(f"  Test  MAE              : {test_mae:.4g}")
print(f"  Test  RMSE             : {test_rmse:.4g}")
print(f"  Overfit gap (Train-Test R²): {train_r2 - test_r2:.4f}")
print("=" * 70)

# Per-target table
rows = []
for i, tgt in enumerate(target_cols):
    yt = y_test_np[:, i]
    yp = y_pred_test[:, i]
    mask = np.abs(yt) > 1e-12
    mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100 if mask.any() else np.nan
    rows.append({
        'target': tgt,
        'R2':     r2_score(yt, yp),
        'MAE':    mean_absolute_error(yt, yp),
        'RMSE':   np.sqrt(mean_squared_error(yt, yp)),
        'MAPE_%': mape,
    })
df_per_target = pd.DataFrame(rows).sort_values('R2', ascending=False)
print("\nPer-target metrics (top 15 by R²):")
print(df_per_target.head(15).to_string(index=False))


if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    _stem_csv = "simple_nn_full_profile"
    _per_csv = EXPORT_DIR / f"{_stem_csv}_per_target_metrics.csv"
    _grp_csv = EXPORT_DIR / f"{_stem_csv}_group_metrics.csv"

    chemistry_serializable = {k: list(v) for k, v in sorted(chemistry_groups.items())}

    def _group_block(cols):
        cols = [c for c in cols if c in target_cols]
        if not cols:
            return None
        idx = [target_cols.index(c) for c in cols]
        yt = y_test_np[:, idx]
        yp = y_pred_test[:, idx]
        return {
            "targets": cols,
            "n_targets": len(cols),
            "test_r2_uniform_avg": float(r2_score(yt, yp, multioutput="uniform_average")),
            "test_mae_uniform_avg": float(mean_absolute_error(yt, yp, multioutput="uniform_average")),
            "test_rmse_uniform_avg": float(
                np.sqrt(mean_squared_error(yt, yp, multioutput="uniform_average"))
            ),
        }

    metrics_by_group = {}
    st_block = _group_block(state_target_cols)
    if st_block is not None:
        metrics_by_group["state_thermo"] = st_block
    for role, cols in sorted(chemistry_groups.items()):
        key = f"species_{role}"
        blk = _group_block(cols)
        if blk is not None:
            metrics_by_group[key] = blk

    df_per_target.to_csv(_per_csv, index=False)
    _gm_rows = []
    for gname, gdata in metrics_by_group.items():
        _gm_rows.append(
            {
                "group": gname,
                "n_targets": gdata["n_targets"],
                "test_r2_uniform_avg": gdata["test_r2_uniform_avg"],
                "test_mae_uniform_avg": gdata["test_mae_uniform_avg"],
                "test_rmse_uniform_avg": gdata["test_rmse_uniform_avg"],
            }
        )
    pd.DataFrame(_gm_rows).to_csv(_grp_csv, index=False)
    print(f"  Exported metrics: {_per_csv.name}, {_grp_csv.name}")

# 9b. AXIAL OVERLAYS — few test runs (Cantera / test vs NN vs x/L)
# y_pred_test is inlet-anchored in §9 (min x/L per run) so curves meet at the left.
# ════════════════════════════════════════════════════════════════════════════
AXIAL_COMPARISON_POSITIONS = [0.25, 0.50, 0.75, 1.00]
AXIAL_PROFILE_N_RUNS = 4
AXIAL_FEED_LABEL_OVERRIDE = None
# AXIAL_PROFILE_RUNS_RANDOM — how test runs are picked for the overlay columns:
#   False (default): always the SAME runs each time — the first N distinct run_id values
#     in the order rows appear in X_test_df (sorted by index / file order, not shuffled).
#   True: a RANDOM subset of N distinct run_id values, reproducible via RANDOM_STATE.
AXIAL_PROFILE_RUNS_RANDOM = True
_AXIAL_STATE_PREFS = [
    "temperature_K",
    "pressure_Pa",
    "velocity_ms",
    "density_kgm3",
    "mean_molecular_weight_kgkmol",
]
_state_axial = [c for c in _AXIAL_STATE_PREFS if c in target_cols]
_spec_axial = [c for c in species_cols if c in target_cols]
AXIAL_PROFILE_TARGETS = _state_axial + _spec_axial

if (
    "relative_position" in X_test_df.columns
    and AXIAL_PROFILE_TARGETS
    and len(X_test_df) > 0
):
    test_run_id = full_run_id.loc[X_test_df.index]
    full_plot_meta = X_test_df[["relative_position"]].copy()
    full_plot_meta["_run_id"] = test_run_id.to_numpy()
    if "reactant_type" in X_test_df.columns and label_encoder is not None:
        try:
            full_plot_meta["_feed"] = label_encoder.inverse_transform(
                X_test_df["reactant_type"].astype(int).to_numpy()
            )
        except Exception:
            full_plot_meta["_feed"] = X_test_df["reactant_type"].astype(str).to_numpy()
    elif "reactant_type" in X_test_df.columns:
        full_plot_meta["_feed"] = X_test_df["reactant_type"].astype(str).to_numpy()
    else:
        full_plot_meta["_feed"] = np.array(["unknown_feed"] * len(full_plot_meta), dtype=object)

    full_actual_df = y_test_df.loc[:, target_cols].copy()
    full_pred_df = pd.DataFrame(y_pred_test, index=X_test_df.index, columns=target_cols)

    _unique_runs = pd.Series(full_plot_meta["_run_id"]).drop_duplicates().to_list()
    _n_pick = int(min(AXIAL_PROFILE_N_RUNS, len(_unique_runs)))
    if _n_pick <= 0:
        selected_profile_runs = []
    elif AXIAL_PROFILE_RUNS_RANDOM:
        _rng_ax = np.random.default_rng(int(RANDOM_STATE))
        selected_profile_runs = list(_rng_ax.choice(_unique_runs, size=_n_pick, replace=False))
    else:
        selected_profile_runs = _unique_runs[:_n_pick]
    _run_pick_mode = (
        f"random subset of {len(_unique_runs)} distinct test run_id (numpy seed={int(RANDOM_STATE)})"
        if AXIAL_PROFILE_RUNS_RANDOM
        else f"fixed order: first {_n_pick} distinct run_id in test DataFrame row order (same choice every run unless data order changes)"
    )
    print(f"[9b] Axial overlay run selection — {_run_pick_mode}")
    print(f"[9b] run_id used: {selected_profile_runs}")
    n_rows = len(AXIAL_PROFILE_TARGETS)
    n_cols = len(selected_profile_runs)

    target_labels = {
        "temperature_K": "Temperature (K)",
        "pressure_Pa": "Pressure (bar)",
        "velocity_ms": "Velocity (m/s)",
        "density_kgm3": "Density (kg/m³)",
        "mean_molecular_weight_kgkmol": "Mean molecular weight (kg/kmol)",
    }
    for _t in _spec_axial:
        if _t not in target_labels:
            _s = _t.replace("Y_lump_", "").replace("Y_", "")
            target_labels[_t] = f"Mass fraction Y ({_s})"

    if n_cols > 0:
        fig_ax, axes = plt.subplots(
            n_rows,
            n_cols,
            figsize=(5.2 * n_cols, 3.2 * n_rows),
            squeeze=False,
        )
        station_rows = []

        for col_idx, run_id in enumerate(selected_profile_runs):
            run_mask = full_plot_meta["_run_id"] == run_id
            run_index = full_plot_meta.index[run_mask]
            _feed_vals = full_plot_meta.loc[run_index, "_feed"]
            feed_label = str(_feed_vals.iloc[0]) if len(_feed_vals) else "unknown_feed"
            if AXIAL_FEED_LABEL_OVERRIDE:
                feed_label = AXIAL_FEED_LABEL_OVERRIDE
            x_rel = full_plot_meta.loc[run_index, "relative_position"].astype(float)
            order = np.argsort(x_rel.to_numpy())
            x_sorted = x_rel.to_numpy()[order]
            sorted_index = run_index[order]

            for row_idx, tgt in enumerate(AXIAL_PROFILE_TARGETS):
                ax = axes[row_idx, col_idx]
                actual = full_actual_df.loc[sorted_index, tgt].to_numpy(dtype=float)
                pred = full_pred_df.loc[sorted_index, tgt].to_numpy(dtype=float)
                if tgt == "pressure_Pa":
                    actual = actual / 1e5
                    pred = pred / 1e5

                ax.plot(x_sorted, actual, "b-", lw=2, label="Cantera / test")
                ax.plot(x_sorted, pred, "r-", lw=2, label="NN prediction (full profile)")

                for station in AXIAL_COMPARISON_POSITIONS:
                    ax.axvline(station, color="k", linestyle="--", lw=1.25, alpha=0.5)

                ax.set_xlim(0, 1.02)
                ax.set_title(
                    f"Run {run_id} | Feed={feed_label} — {target_labels.get(tgt, tgt)}",
                    fontsize=9,
                )
                if row_idx == n_rows - 1:
                    ax.set_xlabel("x/L")
                if col_idx == 0:
                    ax.set_ylabel(target_labels.get(tgt, tgt))
                if row_idx == 0 and col_idx == 0:
                    ax.legend(loc="best", fontsize=8)

                for station in AXIAL_COMPARISON_POSITIONS:
                    nearest_idx = (x_rel - station).abs().idxmin()
                    actual_station = float(full_actual_df.loc[nearest_idx, tgt])
                    pred_station = float(full_pred_df.loc[nearest_idx, tgt])
                    if tgt == "pressure_Pa":
                        actual_station /= 1e5
                        pred_station /= 1e5
                    station_rows.append(
                        {
                            "Run_ID": run_id,
                            "x_over_L_requested": station,
                            "x_over_L_used": float(
                                full_plot_meta.loc[nearest_idx, "relative_position"]
                            ),
                            "Target": target_labels.get(tgt, tgt),
                            "Actual": actual_station,
                            "Predicted": pred_station,
                            "Abs_Error": abs(pred_station - actual_station),
                        }
                    )

        plt.suptitle(
            "Full-profile axial evolution — state + species: Cantera / test vs SimpleNN (test runs)\n"
            + _run_pick_mode,
            y=1.02,
            fontsize=10,
        )
        plt.tight_layout()
        if IF_PLOT_EXPORT:
            FIG_DIR.mkdir(parents=True, exist_ok=True)
            fig_ax.savefig(
                FIG_DIR / "full_profile_cantera_vs_nn_axial_evolution.png",
                dpi=150,
                bbox_inches="tight",
            )
        if IF_PLOT_SHOWN:
            plt.show()
        plt.close(fig_ax)

        df_axial_station_comparison = pd.DataFrame(station_rows)
        print("\nCANTERA / TEST VS NN — selected x/L stations (test runs)")
        print(df_axial_station_comparison.to_string(index=False, float_format=lambda x: f"{x:.5g}"))
    else:
        print("[INFO] No test runs available for axial evolution plot.")
else:
    print("[INFO] Skipping axial overlays (missing relative_position or targets).")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10. ACTUAL VS PREDICTED — STATE & SPECIES (test set)
# ════════════════════════════════════════════════════════════════════════════
# One subplot per target; 4 columns × as many rows as needed (state first, then species).
# Figure export: FIG_DIR / "actual_vs_predicted_scatter_by_target.png" when IF_PLOT_EXPORT.
#
# Small dev runs: if FULL_PROFILE_MAX_ROWS caps data, you may have very few test rows.
# Hexbin then looks empty — we switch to scatter below PARITY_HEXBIN_MIN_POINTS and skip
# the shared density colorbar. Set FULL_PROFILE_MAX_ROWS = None for full test counts.

STATE_LABELS = {
    'temperature_K':                ('Temperature',          'K',     1.0),
    'pressure_Pa':                  ('Pressure',             'bar',   1e-5),
    'velocity_ms':                  ('Velocity',             'm/s',   1.0),
    'density_kgm3':                 ('Density',              'kg/m³', 1.0),
    'mean_molecular_weight_kgkmol': ('Mean MW',              'kg/kmol', 1.0),
    'heat_capacity_cp_JkgK':        ('Cp',                   'J/(kg·K)', 1.0),
    'heat_capacity_cv_JkgK':        ('Cv',                   'J/(kg·K)', 1.0),
    'enthalpy_Jkg':                 ('Enthalpy',             'J/kg',  1.0),
    'thermal_conductivity_WmK':     ('Thermal conductivity', 'W/(m·K)', 1.0),
}

PARITY_HEXBIN_MIN_POINTS = 400


def _label_triplet(target_name):
    if target_name in STATE_LABELS:
        return STATE_LABELS[target_name]
    short = target_name.replace("Y_lump_", "").replace("Y_", "")
    return (short, "", 1.0)


def _axis_title(target_name):
    name, unit, _ = _label_triplet(target_name)
    return f"{name} ({unit})" if unit else name


_state_set = set(state_target_cols)
all_scatter_targets = [c for c in (state_target_cols + species_cols) if c in target_cols]
n_cols = 4
n_panels = len(all_scatter_targets)
n_rows = (n_panels + n_cols - 1) // n_cols if n_panels else 0

if all_scatter_targets and n_rows > 0:
    from matplotlib.colors import LogNorm

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(3.8 * n_cols, 3.2 * n_rows), squeeze=False,
        constrained_layout=True,
    )
    flat_axes = axes.ravel()

    _n_test_pts = int(len(y_test_np))
    _use_hexbin = _n_test_pts >= PARITY_HEXBIN_MIN_POINTS

    # Shared log color scale for hexbin counts (one colorbar for the whole figure).
    _vmax = 1.0
    if _use_hexbin:
        for tgt in all_scatter_targets:
            tgt_idx = target_cols.index(tgt)
            _, _, scale = _label_triplet(tgt)
            yt = y_test_np[:, tgt_idx] * scale
            yp = y_pred_test[:, tgt_idx] * scale
            lo = float(min(yt.min(), yp.min()))
            hi = float(max(yt.max(), yp.max()))
            pad = 0.02 * (hi - lo if hi > lo else 1.0)
            lims = (lo - pad, hi + pad)
            H, _, _ = np.histogram2d(
                yt, yp, bins=40,
                range=[[lims[0], lims[1]], [lims[0], lims[1]]],
            )
            _vmax = max(_vmax, float(H.max()))
        _hb_norm = LogNorm(vmin=1.0, vmax=max(_vmax, 2.0))
    else:
        _hb_norm = None

    _parity_mappable = None

    for i, tgt in enumerate(all_scatter_targets):
        ax = flat_axes[i]
        tgt_idx = target_cols.index(tgt)
        _, _, scale = _label_triplet(tgt)
        yt = y_test_np[:, tgt_idx] * scale
        yp = y_pred_test[:, tgt_idx] * scale

        lo = float(min(yt.min(), yp.min()))
        hi = float(max(yt.max(), yp.max()))
        pad = 0.02 * (hi - lo if hi > lo else 1.0)
        lims = (lo - pad, hi + pad)

        is_state = tgt in _state_set
        if is_state and lo > 0:
            x_band = np.linspace(lims[0], lims[1], 200)
            ax.fill_between(
                x_band, 0.95 * x_band, 1.05 * x_band,
                color="0.85", alpha=0.6, zorder=0, label="±5%",
            )

        if _use_hexbin:
            hb = ax.hexbin(
                yt, yp, gridsize=40, mincnt=1,
                cmap="Blues", norm=_hb_norm, zorder=2,
                extent=(lims[0], lims[1], lims[0], lims[1]),
            )
            if _parity_mappable is None:
                _parity_mappable = hb
        else:
            ax.scatter(
                yt, yp, alpha=0.55, s=28, edgecolors="none", c="b", zorder=2, label="test points",
            )

        ax.plot(lims, lims, "r-", lw=1.5, zorder=3, label="y = x")

        ax.set_xlim(lims)
        ax.set_ylim(lims)
        if is_state:
            ax.set_aspect("equal", adjustable="box")
        else:
            ax.set_aspect("auto")

        r2 = r2_score(yt, yp)
        mae = mean_absolute_error(yt, yp)
        ax.text(
            0.04, 0.96,
            f"R² = {r2:.4f}\nMAE = {mae:.3g}\nn = {len(yt):,}",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.85),
        )

        ax.set_title(_axis_title(tgt), fontsize=10)
        ax.set_xlabel(f"Actual {_axis_title(tgt)}")
        ax.set_ylabel(f"Predicted {_axis_title(tgt)}")

        if i == 0:
            ax.legend(loc="lower right", fontsize=7, frameon=True)

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    _mode = "hexbin + shared log colorbar" if _use_hexbin else f"scatter (n_test={_n_test_pts} < {PARITY_HEXBIN_MIN_POINTS})"
    print(f"[§10] Parity plots: {_mode}  |  FULL_PROFILE_MAX_ROWS={FULL_PROFILE_MAX_ROWS!r}")

    if _use_hexbin and _parity_mappable is not None:
        fig.colorbar(
            _parity_mappable,
            ax=axes,
            shrink=0.72,
            fraction=0.03,
            pad=0.02,
            label="Counts per bin (shared log scale across panels)",
        )

    plt.suptitle(
        "Simple NN — Actual vs Predicted (all state + species targets, test set)",
        y=1.03, fontsize=12,
    )
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            FIG_DIR / "actual_vs_predicted_scatter_by_target.png",
            dpi=150, bbox_inches="tight",
        )
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)
else:
    print("No targets available for parity scatter grid.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10b. RESIDUALS — STATE & SPECIES (test set)
# ════════════════════════════════════════════════════════════════════════════
# Same 4-column layout as §10: residual = predicted − actual vs actual.
# Figure export: FIG_DIR / "residuals_scatter_by_target.png" when IF_PLOT_EXPORT.

_state_set = set(state_target_cols)
all_scatter_targets = [c for c in (state_target_cols + species_cols) if c in target_cols]
n_cols = 4
n_panels = len(all_scatter_targets)
n_rows = (n_panels + n_cols - 1) // n_cols if n_panels else 0

if all_scatter_targets and n_rows > 0:
    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(3.8 * n_cols, 3.0 * n_rows), squeeze=False,
    )
    flat_axes = axes.ravel()

    for i, tgt in enumerate(all_scatter_targets):
        ax = flat_axes[i]
        tgt_idx = target_cols.index(tgt)
        _, unit, scale = _label_triplet(tgt)
        yt = y_test_np[:, tgt_idx] * scale
        yp = y_pred_test[:, tgt_idx] * scale
        res = yp - yt
        sigma = float(np.std(res))
        bias = float(np.mean(res))

        ax.axhline(0, color="r", lw=1.4, zorder=3)
        ax.axhspan(
            -2 * sigma, 2 * sigma, color="0.85", alpha=0.6, zorder=0,
            label=f"±2σ ({2 * sigma:.3g})",
        )
        ax.scatter(yt, res, alpha=0.35, s=10, edgecolors="none", c="b", zorder=2)

        ax.text(
            0.04, 0.96,
            f"bias = {bias:+.3g}\nσ   = {sigma:.3g}\nn   = {len(yt):,}",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="0.7", alpha=0.85),
        )

        ax.set_xlabel(f"Actual {_axis_title(tgt)}")
        ylab = f"Residual ({unit})" if unit else "Residual"
        ax.set_ylabel(ylab)
        ax.set_title(_axis_title(tgt), fontsize=10)
        ax.grid(True, alpha=0.3)
        if i == 0:
            ax.legend(loc="lower right", fontsize=7, frameon=True)

    for j in range(n_panels, len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle(
        "Simple NN — Residuals vs Actual (all state + species targets, test set)",
        y=1.01, fontsize=12,
    )
    plt.tight_layout()
    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            FIG_DIR / "residuals_scatter_by_target.png",
            dpi=150, bbox_inches="tight",
        )
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)
else:
    print("No targets available for residual scatter grid.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10c. PER-TARGET R² BAR CHART
# ════════════════════════════════════════════════════════════════════════════
# Single-glance scoreboard across every target the network predicts, hatch-coded
# by family (state vs species) and sorted descending. R² is clipped to
# [-0.05, 1.0] for plotting so degenerate targets don't compress the scale.

_state_set = set(state_target_cols)
df_bar = df_per_target.copy()
df_bar['family'] = df_bar['target'].apply(lambda t: 'state' if t in _state_set else 'species')
df_bar = df_bar.sort_values('R2', ascending=False).reset_index(drop=True)

# Cap displayed bars to keep the figure readable
MAX_BARS = 40
# White bars + hatch texture by family (hatch renders in edge colour)
BAR_FACE = "white"
EDGE_COL = "0.35"
HATCH_STATE = ""
HATCH_SPECIES = "///"
df_bar_plot = df_bar.head(MAX_BARS).iloc[::-1]  # reverse so highest R² ends up at the top
clipped_r2  = df_bar_plot['R2'].clip(lower=-0.05, upper=1.0)

fig_h = max(5.0, 0.28 * len(df_bar_plot))
fig, ax = plt.subplots(figsize=(9.5, fig_h))
ax.set_facecolor("0.94")  # light panel so white bars read clearly
bars = ax.barh(
    df_bar_plot["target"],
    clipped_r2,
    color=BAR_FACE,
    edgecolor=EDGE_COL,
    linewidth=0.75,
)
for bar, fam in zip(bars, df_bar_plot["family"]):
    bar.set_hatch(HATCH_STATE if fam == "state" else HATCH_SPECIES)
ax.axvline(0.9, color='lime', lw=1.0, ls='--', alpha=0.9, label='R² = 0.9 (good)')
ax.axvline(0.6, color='b', lw=1.0, ls='--', alpha=0.9, label='R² = 0.6 (guide)')
ax.axvline(0.5, color='r', lw=1.0, ls='--', alpha=0.9, label='R² = 0.5 (weak)')
ax.axvline(0.0, color='k', lw=1.0, ls='--', alpha=0.9, label='R² = 0 (naive baseline)')

# Annotate exact R² to the right of each bar
for tgt, r2 in zip(df_bar_plot['target'], df_bar_plot['R2']):
    ax.text(min(max(r2, -0.05), 1.0) + 0.01, tgt, f'{r2:.3f}',
            va='center', ha='left', fontsize=7.5, color='0.2')

ax.set_xlim(-0.1, 1.12)
ax.set_xlabel('Test R²')
ax.set_title(
    f'Per-target test R²  ({len(df_bar_plot)} of {len(df_bar)} targets shown,'
    f' sorted descending)',
    fontsize=11,
)

# Family legend (match bar face, edge, hatch)
from matplotlib.patches import Patch
family_legend = [
    Patch(
        facecolor=BAR_FACE, edgecolor=EDGE_COL, linewidth=0.75,
        hatch=HATCH_STATE, label="state / thermo",
    ),
    Patch(
        facecolor=BAR_FACE, edgecolor=EDGE_COL, linewidth=0.75,
        hatch=HATCH_SPECIES, label="species (Y_*)",
    ),
]
leg1 = ax.legend(handles=family_legend, loc='lower right', fontsize=9, frameon=True)
ax.add_artist(leg1)
ax.legend(loc='center right', fontsize=8, frameon=True)
plt.tight_layout()

if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'per_target_r2_bar.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

print(f"All targets: {len(df_bar)}  |  shown: {len(df_bar_plot)}  |  "
      f"min R²={df_bar['R2'].min():.4f}  median R²={df_bar['R2'].median():.4f}  "
      f"max R²={df_bar['R2'].max():.4f}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 11. EXPORT MODEL ARTIFACTS
# ════════════════════════════════════════════════════════════════════════════
# state_dict + scalers/encoder + JSON manifest for downstream inference.
# Also writes per-target and per-chemistry-group metric tables (CSV).

if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stem = "simple_nn_full_profile"
    run_at = datetime.now().isoformat(timespec="seconds")

    state_path    = EXPORT_DIR / f"{stem}_state_dict.pt"
    scalers_path  = EXPORT_DIR / f"{stem}_scalers.joblib"
    manifest_path = EXPORT_DIR / f"{stem}_manifest.json"
    per_target_csv = EXPORT_DIR / f"{stem}_per_target_metrics.csv"
    group_metrics_csv = EXPORT_DIR / f"{stem}_group_metrics.csv"

    torch.save(model.state_dict(), state_path)
    joblib.dump(
        {"scaler_X": scaler_X, "scaler_y": scaler_y, "label_encoder": label_encoder},
        scalers_path,
    )

    if IF_HYPERPARAM_TUNING:
        tuning_info = {
            "enabled": True,
            "n_trials_completed": int(n_trials_complete),
            "best_val_r2": float(best_val_r2) if best_val_r2 is not None else None,
            "best_params": best_params if best_params is not None else {},
        }
    else:
        tuning_info = {"enabled": False}

    chemistry_serializable = {k: list(v) for k, v in sorted(chemistry_groups.items())}

    def _group_block(cols):
        cols = [c for c in cols if c in target_cols]
        if not cols:
            return None
        idx = [target_cols.index(c) for c in cols]
        yt = y_test_np[:, idx]
        yp = y_pred_test[:, idx]
        return {
            "targets": cols,
            "n_targets": len(cols),
            "test_r2_uniform_avg": float(r2_score(yt, yp, multioutput="uniform_average")),
            "test_mae_uniform_avg": float(mean_absolute_error(yt, yp, multioutput="uniform_average")),
            "test_rmse_uniform_avg": float(
                np.sqrt(mean_squared_error(yt, yp, multioutput="uniform_average"))
            ),
        }

    metrics_by_group = {}
    st_block = _group_block(state_target_cols)
    if st_block is not None:
        metrics_by_group["state_thermo"] = st_block
    for role, cols in sorted(chemistry_groups.items()):
        key = f"species_{role}"
        blk = _group_block(cols)
        if blk is not None:
            metrics_by_group[key] = blk

    df_per_target.to_csv(per_target_csv, index=False)
    _gm_rows = []
    for gname, gdata in metrics_by_group.items():
        _gm_rows.append(
            {
                "group": gname,
                "n_targets": gdata["n_targets"],
                "test_r2_uniform_avg": gdata["test_r2_uniform_avg"],
                "test_mae_uniform_avg": gdata["test_mae_uniform_avg"],
                "test_rmse_uniform_avg": gdata["test_rmse_uniform_avg"],
            }
        )
    pd.DataFrame(_gm_rows).to_csv(group_metrics_csv, index=False)

    manifest = {
        "model": "SimpleNN",
        "workflow": "full_profile",
        "run_level_split": True,
        "run_at": run_at,
        "architecture": {
            "in_features":  int(n_inputs),
            "h1":           int(H1),
            "h2":           int(H2),
            "h3":           int(H3),
            "dropout":      float(DROPOUT),
            "out_features": int(n_outputs),
        },
        "training": {
            "epochs":        int(EPOCHS),
            "batch_size":    int(batch_size),
            "learning_rate": float(LEARNING_RATE),
            "random_state":  int(RANDOM_STATE),
            "scale_targets": bool(SCALE_TARGETS),
            "early_stopped": bool(EARLY_STOPPED),
            "best_test_r2_checkpoint": float(BEST_TEST_R2_CKPT),
            "best_test_r2_epoch": int(BEST_EPOCH_CKPT),
        },
        "feature_cols":      feature_cols,
        "run_cols":          run_cols,
        "train_runs":        int(len(train_runs)),
        "test_runs":         int(len(test_runs)),
        "n_train_rows":      int(len(X_train_df)),
        "n_test_rows":       int(len(X_test_df)),
        "full_profile_max_rows": FULL_PROFILE_MAX_ROWS,
        "state_target_cols": state_target_cols,
        "species_cols":      species_cols,
        "target_cols":       target_cols,
        "chemistry_groups":  chemistry_serializable,
        "metrics_by_group":  metrics_by_group,
        "data_file":         str(DATA_FILE),
        "metrics": {
            "train_r2":  float(train_r2),
            "test_r2":   float(test_r2),
            "train_r2_state": float(train_r2_state),
            "test_r2_state": float(test_r2_state),
            "train_r2_species": float(train_r2_species),
            "test_r2_species": float(test_r2_species),
            "test_mae":  float(test_mae),
            "test_rmse": float(test_rmse),
        },
        "auxiliary_exports": {
            "per_target_metrics_csv": str(per_target_csv),
            "group_metrics_csv": str(group_metrics_csv),
        },
        "tuning": tuning_info,
    }
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print(f"Exported NN: {state_path.name}, {scalers_path.name}, {manifest_path.name}")
    print(f"  + {per_target_csv.name}, {group_metrics_csv.name}")
else:
    print("Model export disabled.")


---

## Summary

- Workflow: inlet design vector → outlet (exit-plane) state + species mass fractions, predicted with a 3-hidden-layer ReLU MLP (dropout between hidden blocks), trained for `EPOCHS` epochs with Adam and `nn.MSELoss` on standardised targets. Section 8 applies `ReduceLROnPlateau`, optional early stopping on stalled test R², then restores the best test-R² checkpoint before metrics and export.
- Overfitting controls: dropout, held-out test split, scalers fit on train only, shuffled minibatches, and train vs test convergence curves (see overview).
- Scope: baseline first, optional Optuna TPE search (Section 6b) over `h1/h2/h3/dropout/learning_rate/batch_size` when `IF_HYPERPARAM_TUNING=True`. Search uses a validation fold (test set held out), median pruning, validation R² objective, then refits the production model on the best hyperparameters before normal training and evaluation. Section 6b-ii renders Optuna figures from a JSON snapshot (re-plot without re-running the study).
- Next steps: compare these test-set metrics with the tree baselines from [Main_4_train_and_evaluate_tree_models_IO.ipynb](Main_4_train_and_evaluate_tree_models_IO.ipynb); optional refinements include Adam weight decay, weighted or huber loss across targets, larger data caps (`FULL_PROFILE_MAX_ROWS=None`), and deployment packaging beyond this training notebook.